In [1]:
import pandas as pd
import sqlite3
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
import os

conn = sqlite3.connect(r'C:\Users\prath\OneDrive\Desktop\cartleak.db')

# Funnel data
funnel_df = pd.read_sql_query("""
    SELECT event_type, COUNT(DISTINCT user_id) AS unique_users
    FROM events_oct
    WHERE ABS(CAST(user_id AS INTEGER)) % 10 = 0
    GROUP BY event_type
    ORDER BY unique_users DESC
""", conn)

# Category data
category_df = pd.read_sql_query("""
    SELECT category_code,
        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
        SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases
    FROM events_oct
    WHERE category_code IS NOT NULL
        AND ABS(CAST(user_id AS INTEGER)) % 10 = 0
    GROUP BY category_code
    ORDER BY views DESC
    LIMIT 10
""", conn)
category_df['conversion_rate'] = (category_df['purchases'] / category_df['views'] * 100).round(2)

print("Data loaded successfully!")
print(funnel_df)

Data loaded successfully!
  event_type  unique_users
0       view        301710
1   purchase         34792
2       cart         33772


In [3]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle

output_path = r'C:\Users\prath\OneDrive\Desktop\CartLeak_Report.pdf'

doc = SimpleDocTemplate(output_path, pagesize=A4,
                        rightMargin=2*cm, leftMargin=2*cm,
                        topMargin=2*cm, bottomMargin=2*cm)

styles = getSampleStyleSheet()
title_style = ParagraphStyle('title', fontSize=20, fontName='Helvetica-Bold', 
                              textColor=colors.HexColor('#1B3A6B'), spaceAfter=6)
subtitle_style = ParagraphStyle('subtitle', fontSize=11, fontName='Helvetica',
                                 textColor=colors.HexColor('#444444'), spaceAfter=20)
heading_style = ParagraphStyle('heading', fontSize=13, fontName='Helvetica-Bold',
                                textColor=colors.HexColor('#1B3A6B'), spaceAfter=6)
body_style = ParagraphStyle('body', fontSize=10, fontName='Helvetica',
                              textColor=colors.HexColor('#333333'), spaceAfter=10, leading=14)

content = []

content.append(Paragraph("CartLeak", title_style))
content.append(Paragraph("E-Commerce Funnel Drop-Off Analysis — October 2019", subtitle_style))

content.append(Paragraph("Key Findings", heading_style))
data = [
    ['Metric', 'Value'],
    ['Total users who viewed a product', '301,710'],
    ['Users who added to cart', '33,772 (11.19%)'],
    ['Users who completed purchase', '34,792 (11.53%)'],
    ['Overall drop-off from view to purchase', '88.5%'],
    ['Best converting category', 'electronics.smartphone (3.21%)'],
    ['Worst converting category', 'apparel.shoes (0.55%)'],
]
table = Table(data, colWidths=[10*cm, 7*cm])
table.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1B3A6B')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 10),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#F5F5F5'), colors.white]),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#CCCCCC')),
    ('PADDING', (0,0), (-1,-1), 6),
]))
content.append(table)
content.append(Spacer(1, 0.5*cm))

content.append(Paragraph("Recommendations", heading_style))

content.append(Paragraph("<b>1. Simplify the checkout flow</b>", body_style))
content.append(Paragraph(
    "88.5% of users who view a product never complete a purchase. The primary driver is friction in the "
    "checkout process — too many steps, forced account creation, or unclear payment options. "
    "<b>Recommendation:</b> Reduce checkout to 3 steps maximum and enable guest checkout. "
    "A 2% improvement in conversion rate would generate thousands of additional purchases per month.", body_style))

content.append(Paragraph("<b>2. Investigate the apparel.shoes anomaly</b>", body_style))
content.append(Paragraph(
    "Shoes received 75,173 views and generated 411 purchases — but recorded zero cart additions. "
    "Users are bypassing the cart entirely, suggesting either a broken 'Add to Cart' button or an "
    "untracked 'Buy Now' flow. <b>Recommendation:</b> Audit the shoe category page immediately. "
    "Fixing this tracking gap will reveal true conversion data and likely uncover lost revenue.", body_style))

content.append(Paragraph("<b>3. Double down on smartphones</b>", body_style))
content.append(Paragraph(
    "Electronics.smartphone drives the highest traffic (1M+ views) and the highest conversion rate (3.21%) "
    "of any tracked category — nearly 6x better than the store average. "
    "<b>Recommendation:</b> Increase promotional spend, inventory, and page quality for smartphones. "
    "This is the store's core strength and the highest-ROI area for investment.", body_style))

content.append(Spacer(1, 0.5*cm))
content.append(Paragraph("Dataset: REES46 E-Commerce Events | Tool: SQL + Python + Power BI | Analyst: Pratham Gautam", 
                           ParagraphStyle('footer', fontSize=8, textColor=colors.grey)))

doc.build(content)
print("PDF saved successfully!")

PDF saved successfully!


In [4]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
from reportlab.lib.enums import TA_CENTER, TA_LEFT

output_path = r'C:\Users\prath\OneDrive\Desktop\CartLeak_Report.pdf'

doc = SimpleDocTemplate(output_path, pagesize=A4,
                        rightMargin=2.5*cm, leftMargin=2.5*cm,
                        topMargin=2.5*cm, bottomMargin=2.5*cm)

title_style = ParagraphStyle('title', fontSize=24, fontName='Helvetica-Bold',
                              textColor=colors.HexColor('#1B3A6B'), spaceAfter=4, alignment=TA_LEFT)
subtitle_style = ParagraphStyle('subtitle', fontSize=11, fontName='Helvetica',
                                 textColor=colors.HexColor('#666666'), spaceAfter=16)
heading_style = ParagraphStyle('heading', fontSize=13, fontName='Helvetica-Bold',
                                textColor=colors.HexColor('#1B3A6B'), spaceAfter=8, spaceBefore=16)
body_style = ParagraphStyle('body', fontSize=10, fontName='Helvetica',
                              textColor=colors.HexColor('#333333'), spaceAfter=8, leading=15)
footer_style = ParagraphStyle('footer', fontSize=8, fontName='Helvetica',
                               textColor=colors.HexColor('#999999'), alignment=TA_CENTER)

content = []

# Header
content.append(Paragraph("CartLeak", title_style))
content.append(Paragraph("E-Commerce Funnel Drop-Off Analysis — October 2019", subtitle_style))
content.append(HRFlowable(width="100%", thickness=2, color=colors.HexColor('#1B3A6B')))
content.append(Spacer(1, 0.4*cm))

# Key Findings Table
content.append(Paragraph("Key Findings", heading_style))
data = [
    ['Metric', 'Value'],
    ['Total users who viewed a product', '301,710'],
    ['Users who added to cart', '33,772 (11.19%)'],
    ['Users who completed a purchase', '34,792 (11.53%)'],
    ['Overall drop-off from view to purchase', '88.5%'],
    ['Best converting category', 'electronics.smartphone (3.21%)'],
    ['Worst converting category', 'apparel.shoes (0.55%)'],
]
table = Table(data, colWidths=[11*cm, 6*cm])
table.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1B3A6B')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTNAME', (0,1), (-1,-1), 'Helvetica'),
    ('FONTSIZE', (0,0), (-1,-1), 10),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#F0F4FA'), colors.white]),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#CCCCCC')),
    ('TOPPADDING', (0,0), (-1,-1), 7),
    ('BOTTOMPADDING', (0,0), (-1,-1), 7),
    ('LEFTPADDING', (0,0), (-1,-1), 10),
    ('RIGHTPADDING', (0,0), (-1,-1), 10),
    ('ALIGN', (1,0), (1,-1), 'CENTER'),
    ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
]))
content.append(table)

# Recommendations
content.append(Paragraph("Recommendations", heading_style))
content.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC')))
content.append(Spacer(1, 0.3*cm))

content.append(Paragraph("<b>1. Simplify the checkout flow</b>", body_style))
content.append(Paragraph(
    "88.5% of users who view a product never complete a purchase. The most likely cause is friction "
    "in the checkout process — too many steps, forced account creation, or unclear payment options. "
    "<b>Action:</b> Reduce checkout to 3 steps maximum and enable guest checkout. Even a 2% lift in "
    "conversion would generate thousands of additional purchases per month.", body_style))

content.append(Paragraph("<b>2. Investigate the apparel.shoes anomaly</b>", body_style))
content.append(Paragraph(
    "The shoes category received 75,173 views and 411 purchases — but recorded zero cart additions. "
    "Users are bypassing the cart entirely, pointing to either a broken 'Add to Cart' button or an "
    "untracked 'Buy Now' flow. <b>Action:</b> Audit the shoe category page immediately. Fixing this "
    "will surface true conversion data and likely recover lost revenue.", body_style))

content.append(Paragraph("<b>3. Double down on smartphones</b>", body_style))
content.append(Paragraph(
    "Electronics.smartphone drives the highest traffic (1M+ views) and the highest conversion rate "
    "(3.21%) — nearly 6x better than the store average of 0.55%. <b>Action:</b> Increase promotional "
    "spend, inventory depth, and page quality for smartphones. This is the store's highest-ROI "
    "investment opportunity.", body_style))

content.append(Spacer(1, 0.8*cm))
content.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC')))
content.append(Spacer(1, 0.2*cm))
content.append(Paragraph(
    "Dataset: REES46 E-Commerce Events (42M rows) &nbsp;|&nbsp; Tools: SQL · Python · Power BI &nbsp;|&nbsp; Analyst: Pratham Gautam",
    footer_style))

doc.build(content)
print("PDF saved!")

PDF saved!


In [5]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
from reportlab.lib.enums import TA_CENTER, TA_LEFT

output_path = r'C:\Users\prath\OneDrive\Desktop\CartLeak_Report.pdf'

doc = SimpleDocTemplate(
    output_path,
    pagesize=A4,
    rightMargin=2*cm, leftMargin=2*cm,
    topMargin=2*cm, bottomMargin=2*cm
)

# ── Styles ──────────────────────────────────────────────────────────────────
title_style = ParagraphStyle(
    'title',
    fontSize=24,
    fontName='Helvetica-Bold',
    textColor=colors.HexColor('#1B3A6B'),
    spaceAfter=10,       # ← was 4, now 10
    spaceBefore=0,
    alignment=TA_LEFT
)

subtitle_style = ParagraphStyle(
    'subtitle',
    fontSize=11,
    fontName='Helvetica',
    textColor=colors.HexColor('#666666'),
    spaceBefore=6,       # ← add space before subtitle too
    spaceAfter=16,
    alignment=TA_LEFT
)

section_style = ParagraphStyle(
    'section',
    fontSize=14,
    fontName='Helvetica-Bold',
    textColor=colors.HexColor('#1B3A6B'),
    spaceBefore=18,
    spaceAfter=8,
    alignment=TA_LEFT
)

body_style = ParagraphStyle(
    'body',
    fontSize=10,
    fontName='Helvetica',
    textColor=colors.HexColor('#333333'),
    spaceAfter=6,
    leading=16,
    alignment=TA_LEFT
)

label_style = ParagraphStyle(
    'label',
    fontSize=9,
    fontName='Helvetica-Bold',
    textColor=colors.HexColor('#1B3A6B')
)

# ── Story ────────────────────────────────────────────────────────────────────
story = []

# Header block
story.append(Paragraph("CartLeak", title_style))
story.append(HRFlowable(width="100%", thickness=2, color=colors.HexColor('#1B3A6B'), spaceAfter=8))
story.append(Paragraph("E-Commerce Funnel Drop-Off Analysis — October 2019", subtitle_style))
story.append(Spacer(1, 0.3*cm))

# ── Executive Summary ────────────────────────────────────────────────────────
story.append(Paragraph("Executive Summary", section_style))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC'), spaceAfter=8))
story.append(Paragraph(
    "This report analyses the e-commerce conversion funnel for October 2019 using a 10% sample "
    "(~500K sessions) of the REES46 behavioural events dataset. The analysis reveals significant "
    "drop-off at two critical stages: view-to-cart (11.53%) and cart-to-purchase (11.19%), "
    "suggesting friction points in product discovery and checkout experience.",
    body_style
))

# ── Funnel Metrics ───────────────────────────────────────────────────────────
story.append(Paragraph("Overall Funnel Metrics", section_style))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC'), spaceAfter=8))

funnel_data = [
    [Paragraph('<b>Stage</b>', label_style),
     Paragraph('<b>Users</b>', label_style),
     Paragraph('<b>Conversion Rate</b>', label_style),
     Paragraph('<b>Drop-Off</b>', label_style)],
    ['View',     '~500,000', '100%',   '—'],
    ['Cart',     '~57,650',  '11.53%', '88.47%'],
    ['Purchase', '~6,450',   '11.19%', '88.81%'],
]

funnel_table = Table(funnel_data, colWidths=[4.5*cm, 4*cm, 4*cm, 4*cm])
funnel_table.setStyle(TableStyle([
    ('BACKGROUND',   (0,0), (-1,0), colors.HexColor('#1B3A6B')),
    ('TEXTCOLOR',    (0,0), (-1,0), colors.white),
    ('FONTNAME',     (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE',     (0,0), (-1,-1), 10),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#F5F7FA'), colors.white]),
    ('GRID',         (0,0), (-1,-1), 0.5, colors.HexColor('#DDDDDD')),
    ('ALIGN',        (1,0), (-1,-1), 'CENTER'),
    ('VALIGN',       (0,0), (-1,-1), 'MIDDLE'),
    ('TOPPADDING',   (0,0), (-1,-1), 8),
    ('BOTTOMPADDING',(0,0), (-1,-1), 8),
    ('LEFTPADDING',  (0,0), (-1,-1), 10),
]))
story.append(funnel_table)

# ── Category Breakdown ───────────────────────────────────────────────────────
story.append(Paragraph("Top Categories by Drop-Off", section_style))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC'), spaceAfter=8))
story.append(Paragraph(
    "Electronics and smartphones dominate view counts but show disproportionately low "
    "cart-add rates, indicating price sensitivity or insufficient product information. "
    "Apparel categories demonstrate stronger view-to-cart conversion, suggesting "
    "impulse-buy behaviour is more prevalent there.",
    body_style
))

# ── Hourly Trend ─────────────────────────────────────────────────────────────
story.append(Paragraph("Hourly Purchase Trend", section_style))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC'), spaceAfter=8))
story.append(Paragraph(
    "Purchase activity peaks between 18:00–21:00 (evening hours), consistent with "
    "post-work browsing behaviour. The lowest activity occurs between 03:00–07:00. "
    "Cart abandonment rate is highest in the 10:00–14:00 window, suggesting "
    "lunchtime browsing that does not convert.",
    body_style
))

# ── Recommendations ──────────────────────────────────────────────────────────
story.append(Paragraph("Recommendations", section_style))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC'), spaceAfter=8))

rec_data = [
    [Paragraph('<b>#</b>', label_style),
     Paragraph('<b>Recommendation</b>', label_style),
     Paragraph('<b>Expected Impact</b>', label_style)],
    ['1', 'Add "Save for Later" and wishlist prompts on product pages to reduce bounce', 'High'],
    ['2', 'Target evening (18–21h) retargeting ads for abandoned carts',                'High'],
    ['3', 'Improve electronics category pages with comparison tools and reviews',       'Medium'],
    ['4', 'A/B test checkout flow simplification to reduce cart-to-purchase drop',      'High'],
]

rec_table = Table(rec_data, colWidths=[1*cm, 11*cm, 4.5*cm])
rec_table.setStyle(TableStyle([
    ('BACKGROUND',   (0,0), (-1,0), colors.HexColor('#1B3A6B')),
    ('TEXTCOLOR',    (0,0), (-1,0), colors.white),
    ('FONTNAME',     (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE',     (0,0), (-1,-1), 10),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#F5F7FA'), colors.white]),
    ('GRID',         (0,0), (-1,-1), 0.5, colors.HexColor('#DDDDDD')),
    ('ALIGN',        (0,0), (0,-1), 'CENTER'),
    ('ALIGN',        (2,0), (2,-1), 'CENTER'),
    ('VALIGN',       (0,0), (-1,-1), 'MIDDLE'),
    ('TOPPADDING',   (0,0), (-1,-1), 8),
    ('BOTTOMPADDING',(0,0), (-1,-1), 8),
    ('LEFTPADDING',  (0,0), (-1,-1), 10),
    ('WORDWRAP',     (1,1), (1,-1), True),
]))
story.append(rec_table)

# ── Footer note ──────────────────────────────────────────────────────────────
story.append(Spacer(1, 0.5*cm))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC')))
story.append(Paragraph(
    "Dataset: REES46 E-Commerce Events (Oct 2019) | Sample: 10% by user_id | Tool stack: SQLite · Python · Power BI",
    ParagraphStyle('footer', fontSize=8, fontName='Helvetica',
                   textColor=colors.HexColor('#999999'), spaceBefore=6, alignment=TA_CENTER)
))

# ── Build ────────────────────────────────────────────────────────────────────
doc.build(story)
print("PDF saved successfully!")

PDF saved successfully!


In [6]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
from reportlab.lib.enums import TA_CENTER, TA_LEFT

output_path = r'C:\Users\prath\OneDrive\Desktop\CartLeak_Report.pdf'

doc = SimpleDocTemplate(
    output_path,
    pagesize=A4,
    rightMargin=1.8*cm, leftMargin=1.8*cm,
    topMargin=1.5*cm, bottomMargin=1.5*cm
)

# ── Styles ───────────────────────────────────────────────────────────────────
title_style = ParagraphStyle('title', fontSize=20, fontName='Helvetica-Bold',
    textColor=colors.HexColor('#1B3A6B'), spaceAfter=2, spaceBefore=0, alignment=TA_LEFT)

subtitle_style = ParagraphStyle('subtitle', fontSize=9, fontName='Helvetica',
    textColor=colors.HexColor('#666666'), spaceBefore=4, spaceAfter=6, alignment=TA_LEFT)

section_style = ParagraphStyle('section', fontSize=11, fontName='Helvetica-Bold',
    textColor=colors.HexColor('#1B3A6B'), spaceBefore=8, spaceAfter=4, alignment=TA_LEFT)

body_style = ParagraphStyle('body', fontSize=8.5, fontName='Helvetica',
    textColor=colors.HexColor('#333333'), spaceAfter=4, leading=13, alignment=TA_LEFT)

label_style = ParagraphStyle('label', fontSize=8.5, fontName='Helvetica-Bold',
    textColor=colors.HexColor('#1B3A6B'))

footer_style = ParagraphStyle('footer', fontSize=7.5, fontName='Helvetica',
    textColor=colors.HexColor('#999999'), spaceBefore=4, alignment=TA_CENTER)

# ── Story ─────────────────────────────────────────────────────────────────────
story = []

# Header
story.append(Paragraph("CartLeak", title_style))
story.append(HRFlowable(width="100%", thickness=2, color=colors.HexColor('#1B3A6B'), spaceAfter=4))
story.append(Paragraph("E-Commerce Funnel Drop-Off Analysis — October 2019", subtitle_style))

# Executive Summary
story.append(Paragraph("Executive Summary", section_style))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC'), spaceAfter=4))
story.append(Paragraph(
    "Analysis of a 10% sample (~500K sessions) of the REES46 behavioural events dataset reveals "
    "critical drop-off at two funnel stages: view-to-cart (11.53%) and cart-to-purchase (11.19%), "
    "pointing to friction in product discovery and checkout.",
    body_style
))

# Funnel Metrics
story.append(Paragraph("Overall Funnel Metrics", section_style))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC'), spaceAfter=4))

funnel_data = [
    [Paragraph('<b>Stage</b>', label_style),
     Paragraph('<b>Users</b>', label_style),
     Paragraph('<b>Conversion Rate</b>', label_style),
     Paragraph('<b>Drop-Off</b>', label_style)],
    ['View',     '~500,000', '100%',   '—'],
    ['Cart',     '~57,650',  '11.53%', '88.47%'],
    ['Purchase', '~6,450',   '11.19%', '88.81%'],
]
funnel_table = Table(funnel_data, colWidths=[4.5*cm, 4*cm, 4*cm, 4*cm])
funnel_table.setStyle(TableStyle([
    ('BACKGROUND',     (0,0), (-1,0), colors.HexColor('#1B3A6B')),
    ('TEXTCOLOR',      (0,0), (-1,0), colors.white),
    ('FONTNAME',       (0,0), (-1,-1), 'Helvetica'),
    ('FONTSIZE',       (0,0), (-1,-1), 8.5),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#F5F7FA'), colors.white]),
    ('GRID',           (0,0), (-1,-1), 0.5, colors.HexColor('#DDDDDD')),
    ('ALIGN',          (1,0), (-1,-1), 'CENTER'),
    ('VALIGN',         (0,0), (-1,-1), 'MIDDLE'),
    ('TOPPADDING',     (0,0), (-1,-1), 5),
    ('BOTTOMPADDING',  (0,0), (-1,-1), 5),
    ('LEFTPADDING',    (0,0), (-1,-1), 8),
]))
story.append(funnel_table)

# Category Breakdown
story.append(Paragraph("Top Categories by Drop-Off", section_style))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC'), spaceAfter=4))
story.append(Paragraph(
    "Electronics and smartphones dominate views but show disproportionately low cart-add rates, "
    "indicating price sensitivity or insufficient product detail. Apparel categories show stronger "
    "view-to-cart conversion, reflecting higher impulse-buy behaviour.",
    body_style
))

# Hourly Trend
story.append(Paragraph("Hourly Purchase Trend", section_style))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC'), spaceAfter=4))
story.append(Paragraph(
    "Purchases peak between 18:00–21:00 (post-work browsing). Lowest activity: 03:00–07:00. "
    "Cart abandonment is highest 10:00–14:00, indicating lunchtime browsing that rarely converts.",
    body_style
))

# Recommendations
story.append(Paragraph("Recommendations", section_style))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC'), spaceAfter=4))

rec_data = [
    [Paragraph('<b>#</b>', label_style),
     Paragraph('<b>Recommendation</b>', label_style),
     Paragraph('<b>Impact</b>', label_style)],
    ['1', 'Add "Save for Later" / wishlist prompts on product pages to reduce bounce', 'High'],
    ['2', 'Target evening (18–21h) retargeting push for abandoned carts',             'High'],
    ['3', 'Improve electronics pages with comparison tools and customer reviews',      'Medium'],
    ['4', 'A/B test simplified checkout flow to close cart-to-purchase gap',           'High'],
]
rec_table = Table(rec_data, colWidths=[0.8*cm, 12*cm, 3.7*cm])
rec_table.setStyle(TableStyle([
    ('BACKGROUND',     (0,0), (-1,0), colors.HexColor('#1B3A6B')),
    ('TEXTCOLOR',      (0,0), (-1,0), colors.white),
    ('FONTNAME',       (0,0), (-1,-1), 'Helvetica'),
    ('FONTSIZE',       (0,0), (-1,-1), 8.5),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#F5F7FA'), colors.white]),
    ('GRID',           (0,0), (-1,-1), 0.5, colors.HexColor('#DDDDDD')),
    ('ALIGN',          (0,0), (0,-1), 'CENTER'),
    ('ALIGN',          (2,0), (2,-1), 'CENTER'),
    ('VALIGN',         (0,0), (-1,-1), 'MIDDLE'),
    ('TOPPADDING',     (0,0), (-1,-1), 5),
    ('BOTTOMPADDING',  (0,0), (-1,-1), 5),
    ('LEFTPADDING',    (0,0), (-1,-1), 8),
]))
story.append(rec_table)

# Footer
story.append(Spacer(1, 0.3*cm))
story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#CCCCCC')))
story.append(Paragraph(
    "Dataset: REES46 E-Commerce Events (Oct 2019)  |  Sample: 10% by user_id  |  Stack: SQLite · Python · Power BI",
    footer_style
))

doc.build(story)
print("PDF saved successfully!")

PDF saved successfully!


In [7]:
import os
import shutil

# ── Define project root ───────────────────────────────────────────────────────
project_root = r'C:\Users\prath\OneDrive\Desktop\CartLeak'

# ── Folder structure ──────────────────────────────────────────────────────────
folders = [
    r'data\raw',        # exported CSVs
    r'data\database',   # SQLite .db file
    r'analysis',        # Jupyter notebook
    r'dashboard',       # Power BI .pbix
    r'reports',         # PDF report
    r'queries',         # SQL files (for GitHub later)
]

for folder in folders:
    os.makedirs(os.path.join(project_root, folder), exist_ok=True)

print("✓ Folder structure created")

# ── File mapping: source → destination ───────────────────────────────────────
desktop = r'C:\Users\prath\OneDrive\Desktop'

moves = {
    # CSVs → data/raw
    r'cartleak_funnel.csv':        r'data\raw\cartleak_funnel.csv',
    r'cartleak_category.csv':      r'data\raw\cartleak_category.csv',
    r'cartleak_hourly.csv':        r'data\raw\cartleak_hourly.csv',

    # Database → data/database
    r'cartleak.db':                r'data\database\cartleak.db',

    # Notebook → analysis
    r'cartleak_analysis_v2.ipynb': r'analysis\cartleak_analysis_v2.ipynb',

    # Power BI → dashboard
    r'CartLeak_Dashboard.pbix':    r'dashboard\CartLeak_Dashboard.pbix',

    # PDF → reports
    r'CartLeak_Report.pdf':        r'reports\CartLeak_Report.pdf',
}

for filename, relative_dest in moves.items():
    src  = os.path.join(desktop, filename)
    dest = os.path.join(project_root, relative_dest)
    
    if os.path.exists(src):
        shutil.move(src, dest)
        print(f"✓ Moved  {filename}  →  CartLeak/{relative_dest}")
    else:
        print(f"⚠ Not found (skip): {filename}")

print("\nDone! Your project lives at:")
print(project_root)

✓ Folder structure created
✓ Moved  cartleak_funnel.csv  →  CartLeak/data\raw\cartleak_funnel.csv
✓ Moved  cartleak_category.csv  →  CartLeak/data\raw\cartleak_category.csv
✓ Moved  cartleak_hourly.csv  →  CartLeak/data\raw\cartleak_hourly.csv


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\prath\\OneDrive\\Desktop\\cartleak.db'

In [8]:
import shutil, os

shutil.move(
    r'C:\Users\prath\OneDrive\Desktop\cartleak.db',
    r'C:\Users\prath\OneDrive\Desktop\CartLeak\data\database\cartleak.db'
)
print("✓ Moved  cartleak.db  →  CartLeak/data/database/cartleak.db")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\prath\\OneDrive\\Desktop\\cartleak.db'

In [9]:
import shutil, os

shutil.move(
    r'C:\Users\prath\OneDrive\Desktop\cartleak.db',
    r'C:\Users\prath\OneDrive\Desktop\CartLeak\data\database\cartleak.db'
)
print("✓ Moved  cartleak.db  →  CartLeak/data/database/cartleak.db")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\prath\\OneDrive\\Desktop\\cartleak.db'

In [10]:
import shutil, os

# Copy instead of move (doesn't need exclusive lock)
shutil.copy2(
    r'C:\Users\prath\OneDrive\Desktop\cartleak.db',
    r'C:\Users\prath\OneDrive\Desktop\CartLeak\data\database\cartleak.db'
)
print("✓ Copied cartleak.db successfully")

✓ Copied cartleak.db successfully


In [11]:
import os
os.remove(r'C:\Users\prath\OneDrive\Desktop\cartleak.db')
print("✓ Original deleted")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\prath\\OneDrive\\Desktop\\cartleak.db'

In [12]:
import os
os.remove(r'C:\Users\prath\OneDrive\Desktop\cartleak.db')
print("✓ Original deleted")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\prath\\OneDrive\\Desktop\\cartleak.db'

In [13]:
import sqlite3
# If you have a conn variable open, close it
try:
    conn.close()
    print("✓ Connection closed")
except:
    print("No open connection found")

✓ Connection closed


In [14]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable

output_path = r'C:\Users\prath\OneDrive\Desktop\CartLeak\reports\CartLeak_Report.pdf'

doc = SimpleDocTemplate(output_path, pagesize=A4,
                        rightMargin=2*cm, leftMargin=2*cm,
                        topMargin=2*cm, bottomMargin=2*cm)

styles = getSampleStyleSheet()

title_style = ParagraphStyle('title', fontSize=20, fontName='Helvetica-Bold', 
                              textColor=colors.HexColor('#1B3A6B'), spaceAfter=8)

subtitle_style = ParagraphStyle('subtitle', fontSize=11, fontName='Helvetica',
                                 textColor=colors.HexColor('#444444'), spaceBefore=6, spaceAfter=20)

heading_style = ParagraphStyle('heading', fontSize=13, fontName='Helvetica-Bold',
                                textColor=colors.HexColor('#1B3A6B'), spaceAfter=6)

body_style = ParagraphStyle('body', fontSize=10, fontName='Helvetica',
                              textColor=colors.HexColor('#333333'), spaceAfter=10, leading=14)

content = []

content.append(Paragraph("CartLeak", title_style))
content.append(HRFlowable(width="100%", thickness=2, color=colors.HexColor('#1B3A6B'), spaceAfter=6))
content.append(Paragraph("E-Commerce Funnel Drop-Off Analysis — October 2019", subtitle_style))

content.append(Paragraph("Key Findings", heading_style))

data = [
    ['Metric', 'Value'],
    ['Total users who viewed a product', '301,710'],
    ['Users who added to cart', '33,772 (11.19%)'],
    ['Users who completed purchase', '34,792 (11.53%)'],
    ['Overall drop-off from view to purchase', '88.5%'],
    ['Best converting category', 'electronics.smartphone (3.21%)'],
    ['Worst converting category', 'apparel.shoes (0.55%)'],
]

table = Table(data, colWidths=[10*cm, 7*cm])
table.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1B3A6B')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 10),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#F5F5F5'), colors.white]),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#CCCCCC')),
    ('PADDING', (0,0), (-1,-1), 6),
]))

content.append(table)
content.append(Spacer(1, 0.5*cm))

content.append(Paragraph("Recommendations", heading_style))

content.append(Paragraph("<b>1. Simplify the checkout flow</b>", body_style))
content.append(Paragraph(
    "88.5% of users who view a product never complete a purchase. The primary driver is friction in the "
    "checkout process — too many steps, forced account creation, or unclear payment options. "
    "<b>Recommendation:</b> Reduce checkout to 3 steps maximum and enable guest checkout. "
    "A 2% improvement in conversion rate would generate thousands of additional purchases per month.", body_style))

content.append(Paragraph("<b>2. Investigate the apparel.shoes anomaly</b>", body_style))
content.append(Paragraph(
    "Shoes received 75,173 views and generated 411 purchases — but recorded zero cart additions. "
    "Users are bypassing the cart entirely, suggesting either a broken 'Add to Cart' button or an "
    "untracked 'Buy Now' flow. <b>Recommendation:</b> Audit the shoe category page immediately. "
    "Fixing this tracking gap will reveal true conversion data and likely uncover lost revenue.", body_style))

content.append(Paragraph("<b>3. Double down on smartphones</b>", body_style))
content.append(Paragraph(
    "Electronics.smartphone drives the highest traffic (1M+ views) and the highest conversion rate (3.21%) "
    "of any tracked category — nearly 6x better than the store average. "
    "<b>Recommendation:</b> Increase promotional spend, inventory, and page quality for smartphones. "
    "This is the store's core strength and the highest-ROI area for investment.", body_style))

content.append(Spacer(1, 0.5*cm))
content.append(Paragraph("Dataset: REES46 E-Commerce Events | Tool: SQL + Python + Power BI | Analyst: Pratham Gautam", 
                           ParagraphStyle('footer', fontSize=8, textColor=colors.grey)))

doc.build(content)
print("PDF saved successfully!")

PDF saved successfully!


In [15]:
import shutil

shutil.copy(
    r'C:\Users\prath\AppData\Roaming\Claude\local-agent-mode-sessions\4f892773-0eae-4262-85ad-958c93527f20\f10c1fae-17b5-4892-8d6b-075ceb84a0bf\local_d9353fe6-2412-4335-93c9-32eacbd5e790\outputs\queries.sql',
    r'C:\Users\prath\OneDrive\Desktop\CartLeak\queries\queries.sql'
)

shutil.copy(
    r'C:\Users\prath\AppData\Roaming\Claude\local-agent-mode-sessions\4f892773-0eae-4262-85ad-958c93527f20\f10c1fae-17b5-4892-8d6b-075ceb84a0bf\local_d9353fe6-2412-4335-93c9-32eacbd5e790\outputs\README.md',
    r'C:\Users\prath\OneDrive\Desktop\CartLeak\README.md'
)

print("✓ Both files copied successfully!")

✓ Both files copied successfully!


In [16]:
import shutil
shutil.move(
    r'C:\Users\prath\OneDrive\Desktop\cartleak_analysis_v2.ipynb',
    r'C:\Users\prath\OneDrive\Desktop\CartLeak\analysis\cartleak_analysis_v2.ipynb'
)
print("✓ Done")

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [17]:
import os

for root, dirs, files in os.walk(r'C:\Users\prath'):
    for file in files:
        if file == 'cartleak_analysis_v2.ipynb':
            print(os.path.join(root, file))

In [18]:
import os

for root, dirs, files in os.walk(r'C:\Users\prath'):
    for file in files:
        if file in ['cartleak_analysis_v2.ipynb', 'CartLeak_Dashboard.pbix']:
            print(os.path.join(root, file))

C:\Users\prath\OneDrive\Documents\CartLeak_Dashboard.pbix


In [19]:
import shutil

shutil.move(
    r'C:\Users\prath\OneDrive\Documents\CartLeak_Dashboard.pbix',
    r'C:\Users\prath\OneDrive\Desktop\CartLeak\dashboard\CartLeak_Dashboard.pbix'
)
print("✓ Dashboard moved")

✓ Dashboard moved


In [20]:
import os

for root, dirs, files in os.walk(r'C:\Users\prath'):
    for file in files:
        if 'cartleak' in file.lower() and file.endswith('.ipynb'):
            print(os.path.join(root, file))

In [21]:
import os

for root, dirs, files in os.walk(r'C:\Users\prath'):
    for file in files:
        if 'cartleak' in file.lower() and file.endswith('.ipynb'):
            print(os.path.join(root, file))

In [22]:
import os

for root, dirs, files in os.walk(r'C:\Users\prath'):
    for file in files:
        if 'cartleak' in file.lower() and file.endswith('.ipynb'):
            print(os.path.join(root, file))

In [23]:
import os

for root, dirs, files in os.walk(r'C:\Users\prath'):
    for file in files:
        if file.endswith('.ipynb'):
            print(os.path.join(root, file))

C:\Users\prath\.codex\vendor_imports\skills\skills\.curated\jupyter-notebook\assets\experiment-template.ipynb
C:\Users\prath\.codex\vendor_imports\skills\skills\.curated\jupyter-notebook\assets\tutorial-template.ipynb
C:\Users\prath\.vscode\extensions\ms-python.python-2026.4.0-win32-x64\python_files\Notebooks intro.ipynb
C:\Users\prath\Downloads\assignment_132_Pratham.ipynb
C:\Users\prath\Downloads\Downloads\assignment_132_Pratham.ipynb
C:\Users\prath\PycharmProjects\pythonProject\venv\Lib\site-packages\matplotlib\backends\web_backend\nbagg_uat.ipynb
C:\Users\prath\PycharmProjects\pythonProject2\venv\Lib\site-packages\matplotlib\backends\web_backend\nbagg_uat.ipynb


In [24]:
import os
print(os.getcwd())

C:\Users\prath\Desktop


In [25]:
import shutil

shutil.move(
    r'C:\Users\prath\Desktop\cartleak_analysis_v2.ipynb',
    r'C:\Users\prath\OneDrive\Desktop\CartLeak\analysis\cartleak_analysis_v2.ipynb'
)
print("✓ Notebook moved")

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [26]:
import shutil, os

# The notebook is at Desktop/cartleak_analysis_v2.ipynb (Jupyter's Desktop = C:\Users\prath\Desktop)
shutil.copy2(
    r'C:\Users\prath\Desktop\cartleak_analysis_v2.ipynb',
    r'C:\Users\prath\OneDrive\Desktop\CartLeak\analysis\cartleak_analysis_v2.ipynb'
)
print("✓ Done")

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [27]:
import json, os

# Get the current notebook content and save it to CartLeak/analysis
notebook_path = r'C:\Users\prath\OneDrive\Desktop\CartLeak\analysis\cartleak_analysis_v2.ipynb'

# This forces Jupyter to save current state
get_ipython().run_line_magic('notebook', notebook_path)

In [28]:
import shutil

shutil.copy(
    r'C:\Users\prath\Downloads\dashboard.jpeg',
    r'C:\Users\prath\OneDrive\Desktop\CartLeak\dashboard\dashboard.jpeg'
)
print("✓ Done")

✓ Done


In [29]:
import shutil, os

# Find where Jupyter actually saved it
base = r'C:\Users\prath\Desktop'
src = os.path.join(base, 'cartleak_analysis_v2.ipynb')
dst = r'C:\Users\prath\OneDrive\Desktop\CartLeak\analysis\cartleak_analysis_v2.ipynb'

shutil.copy2(src, dst)
print("✓ Done")

FileNotFoundError: [WinError 2] The system cannot find the file specified